Instalação de pacotes, seguido de importação de bibliotecas e funções necessárias.

In [1]:
# Instalar ou atualizar biblioteca necessária.
!pip install -q -U openai

# Importar bibliotecas.
import pandas as pd
import os
from google.colab import userdata
from openai import OpenAI

In [2]:
# Carregar as questões a serem usadas, cuja lógica está em outro arquivo.
# Realizar download do arquivo direto do GitHub.
!wget https://raw.githubusercontent.com/eduoududu/juridico/refs/heads/main/questions.ipynb -O questions.ipynb

# Executar o notebook de base na íntegra.
%run questions.ipynb

--2026-04-03 04:48:49--  https://raw.githubusercontent.com/eduoududu/juridico/refs/heads/main/questions.ipynb
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6565 (6.4K) [text/plain]
Saving to: ‘questions.ipynb’

questions.ipynb     100%[===================>]   6.41K  --.-KB/s    in 0s      

2026-04-03 04:48:49 (83.0 MB/s) - ‘questions.ipynb’ saved [6565/6565]



Configurar modelo do groq a ser usado.

In [ ]:
# Recuperar a chave da API de forma segura, armazenada no Secrets do Google Colab.
# Tal chave previamente criada no Colnsole do Groq, site consele.groq.com
groq_api_key = userdata.get('groq_api_key')

# Inicializar o cliente da API.
client_ai = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=groq_api_key
)

# Modelo gpt-oss de 120 bilhões de parâmetros.
model_id = 'openai/gpt-oss-120b'

# Criar uma lista vazia, para guardar as respostas, por questão de performance.
gpt_responses = []

# Repetição para percorrer todo Dataframe.
for index, row in df_questions_and_guidelines.iterrows():
    # preenchimento dos parâmetros da pergunta, com base na linha corrente.
    questao = row['question']
    papel = row['system']
    categoria = row['category']
    contexto = row['statement']
    instrucao = row['turns']

    # Preparação do prompt em Markdown
    prompt_usuario = f"""
    ### PAPEL:
    {papel}

    ### ÁREA DE ATUAÇÃO:
    {categoria}

    ### CONTEXTO:
    {contexto}

    ### INSTRUÇÃO:
    {instrucao}
    """
    response = client_ai.chat.completions.create(
        model= model_id,
        messages=[
          # Por questões de sintaxes informo a role, pois é um campo obrigatório,
          # porém o conteúdo é somente o do Markdown.
          {"role": "user", "content": prompt_usuario}
        ],
        temperature=0.1 # Conservador.
    )
    response = response.choices[0].message.content

    # Armazenar o resultado em uma lista.
    gpt_responses.append({"question": questao, "response": response})
    print(f"Questão {questao} processada com sucesso.")

# Fechar conexão com a IA (somente se você a usou ativamente)
client_ai.close()

# Para melhor visualização, converter as respostas em um Dataframe.
df_gpt_response = pd.DataFrame(gpt_responses)

Questão 31 processada com sucesso.
Questão 32 processada com sucesso.
Questão 33 processada com sucesso.
Questão 34 processada com sucesso.


In [ ]:
df_questions_and_guidelines.head(1)